<a href="https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score

# ========================================================================
# PART 1 — BUILD RAW DATA
# ========================================================================
rng = np.random.default_rng(42)
N_CLIENTS = 220
MONTHS = 6
rows = []

for client_id in range(N_CLIENTS):
    base_engagement = rng.normal(0, 1)
    plan_type = rng.choice(["basic", "pro", "enterprise"])
    tenure_start = rng.integers(1, 48)
    true_risk = -0.6 * base_engagement + rng.normal(0, 1.2)
    spend_prev = max(20, rng.normal(150, 40))
    for month in range(MONTHS):
        tenure_months = tenure_start + month
        logins_past90d = max(0, rng.normal(30 + 8 * base_engagement, 8))
        support_tickets_past90d = max(0, int(rng.poisson(2 - 0.5 * base_engagement)))
        avg_spend_past90d = max(10, spend_prev + rng.normal(0, 10))
        churn_logit = 0.15 * true_risk - 0.01 * tenure_months + rng.normal(0, 1)
        churn_next_month = int(churn_logit > 1.1)
        spend_next = avg_spend_past90d * (0.15 if churn_next_month else rng.normal(1.0, 0.08))
        spend_drop_pct = (avg_spend_past90d - spend_next) / avg_spend_past90d
        vendor_risk_score = np.clip(0.8 * churn_next_month + rng.normal(0, 0.15), 0, 1)
        rows.append(dict(client_id=client_id, month=month, plan_type=plan_type,
            tenure_months=tenure_months, logins_past90d=logins_past90d,
            support_tickets_past90d=support_tickets_past90d,
            avg_spend_past90d=avg_spend_past90d, spend_drop_pct=spend_drop_pct,
            vendor_risk_score=vendor_risk_score, churn_next_month=churn_next_month))
        spend_prev = avg_spend_past90d

raw = pd.DataFrame(rows)
print("raw built:", raw.shape)

raw built: (1320, 10)


In [ ]:
# ========================================================================
# PART 2 — BUILD FEATURE VECTOR
# ========================================================================
df = raw.copy()
df = pd.get_dummies(df, columns=["plan_type"], drop_first=True)
df["spend_per_login"] = df["avg_spend_past90d"] / df["logins_past90d"].clip(lower=1)

numeric_cols = ["tenure_months", "logins_past90d", "support_tickets_past90d", "avg_spend_past90d", "spend_per_login"]
for c in numeric_cols:
    flag = c + "_was_missing"
    df[flag] = df[c].isna().astype(int)
    df[c] = df[c].fillna(0)

LEAK_COLS = ["spend_drop_pct", "vendor_risk_score"]
HONEST_FEATURES = ["tenure_months", "logins_past90d", "support_tickets_past90d",
                    "avg_spend_past90d", "spend_per_login", "plan_type_enterprise", "plan_type_pro"]
LEAKY_FEATURES = HONEST_FEATURES + LEAK_COLS

print("df built:", df.shape)
print("HONEST_FEATURES:", HONEST_FEATURES)
print("LEAKY_FEATURES:", LEAKY_FEATURES)

df built: (1320, 17)
HONEST_FEATURES: ['tenure_months', 'logins_past90d', 'support_tickets_past90d', 'avg_spend_past90d', 'spend_per_login', 'plan_type_enterprise', 'plan_type_pro']
LEAKY_FEATURES: ['tenure_months', 'logins_past90d', 'support_tickets_past90d', 'avg_spend_past90d', 'spend_per_login', 'plan_type_enterprise', 'plan_type_pro', 'spend_drop_pct', 'vendor_risk_score']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
feature_notes = """
FEATURE NOTES
=============
tenure_months
  meaning: months since signup, as of the snapshot
  missing: fill 0 + tenure_months_was_missing flag
  available before prediction moment: yes - known at snapshot time

logins_past90d
  meaning: login count in the trailing 90 days
  missing: fill 0 + logins_past90d_was_missing flag
  available before prediction moment: yes - window closes before label window opens

support_tickets_past90d
  meaning: support tickets opened, trailing 90 days
  missing: fill 0 + support_tickets_past90d_was_missing flag
  available before prediction moment: yes - trailing window

avg_spend_past90d
  meaning: average monthly spend, trailing 90 days
  missing: fill 0 + avg_spend_past90d_was_missing flag
  available before prediction moment: yes - trailing window

spend_per_login
  meaning: engineered ratio, spend / max(logins, 1)
  missing: inherits fills from its inputs
  available before prediction moment: yes - derived only from trailing-window features

plan_type_pro / plan_type_enterprise
  meaning: one-hot plan tier (basic is the reference level)
  missing: never missing (categorical, always populated)
  available before prediction moment: yes - static attribute

spend_drop_pct (EXCLUDED)
  meaning: percent drop from this period's spend to NEXT period's spend
  missing: n/a
  available before prediction moment: no - built from a value that does not exist yet at prediction time

vendor_risk_score (EXCLUDED)
  meaning: an existing risk-engine's score
  missing: n/a
  available before prediction moment: exists in time, but encodes someone else's decision correlated with the label - circular by construction
"""
print(feature_notes)


FEATURE NOTES
tenure_months
  meaning: months since signup, as of the snapshot
  missing: fill 0 + tenure_months_was_missing flag
  available before prediction moment: yes - known at snapshot time

logins_past90d
  meaning: login count in the trailing 90 days
  missing: fill 0 + logins_past90d_was_missing flag
  available before prediction moment: yes - window closes before label window opens

support_tickets_past90d
  meaning: support tickets opened, trailing 90 days
  missing: fill 0 + support_tickets_past90d_was_missing flag
  available before prediction moment: yes - trailing window

avg_spend_past90d
  meaning: average monthly spend, trailing 90 days
  missing: fill 0 + avg_spend_past90d_was_missing flag
  available before prediction moment: yes - trailing window

spend_per_login
  meaning: engineered ratio, spend / max(logins, 1)
  missing: inherits fills from its inputs
  available before prediction moment: yes - derived only from trailing-window features

plan_type_pro / plan_

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:


# ========================================================================
# PART 3 — THE LEAKAGE HUNT (no nested function — everything inline)
# ========================================================================
y = df["churn_next_month"].values
groups = df["client_id"].values
BASE_RATE = y.mean()
print("\nBase rate (naive baseline): %.3f" % BASE_RATE)
print("Rows:", len(df), "| Clients (groups):", df["client_id"].nunique())

# --- Test A, part 1: honest features, grouped split ---
cv = GroupKFold(n_splits=5)
X = df[HONEST_FEATURES].values
scores = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
                          X, y, cv=cv, groups=groups, scoring="roc_auc")
honest_group = scores.mean()
print("\n--- Test A: without suspects vs with suspects (grouped split) ---")
print("Honest features, grouped split   AUC = %.3f  folds=%s" % (honest_group, np.round(scores, 3)))

# --- Test A, part 2: leaky features, grouped split ---
cv = GroupKFold(n_splits=5)
X = df[LEAKY_FEATURES].values
scores = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
                          X, y, cv=cv, groups=groups, scoring="roc_auc")
leaky_group = scores.mean()
print("Leaky features,  grouped split   AUC = %.3f  folds=%s" % (leaky_group, np.round(scores, 3)))
print("-> AUC moved %.3f -> %.3f when spend_drop_pct / vendor_risk_score were added back. "
      "A jump toward 1.0 is a confession, not a win." % (honest_group, leaky_group))

# --- Test B: honest features, random split ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
X = df[HONEST_FEATURES].values
scores = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
                          X, y, cv=cv, scoring="roc_auc")
honest_random = scores.mean()
print("\n--- Test B: random split vs grouped split (honest features only) ---")
print("Honest features, random split   AUC = %.3f  folds=%s" % (honest_random, np.round(scores, 3)))
gap = honest_random - honest_group
print("-> gap (random - grouped) = %+.3f. A large positive gap would mean the model "
      "memorizes client identity; a small gap means the grouped number is trustworthy on its own." % gap)

# --- Test C: feature importance sanity check ---
rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)
rf.fit(df[LEAKY_FEATURES], y)
importances = pd.Series(rf.feature_importances_, index=LEAKY_FEATURES).sort_values(ascending=False)
print("\n--- Test C: feature importance sanity check on the leaky model ---")
print(importances.to_string())
print("\nTop feature (%s) alone carries %.1f%% of total importance — a single feature "
      "towering over the rest, exactly the symptom to investigate rather than celebrate."
      % (importances.index[0], importances.iloc[0] * 100))


Base rate (naive baseline): 0.098
Rows: 1320 | Clients (groups): 220

--- Test A: without suspects vs with suspects (grouped split) ---
Honest features, grouped split   AUC = 0.572  folds=[0.554 0.531 0.594 0.538 0.643]
Leaky features,  grouped split   AUC = 1.000  folds=[1. 1. 1. 1. 1.]
-> AUC moved 0.572 -> 1.000 when spend_drop_pct / vendor_risk_score were added back. A jump toward 1.0 is a confession, not a win.

--- Test B: random split vs grouped split (honest features only) ---
Honest features, random split   AUC = 0.537  folds=[0.479 0.518 0.585 0.617 0.483]
-> gap (random - grouped) = -0.035. A large positive gap would mean the model memorizes client identity; a small gap means the grouped number is trustworthy on its own.

--- Test C: feature importance sanity check on the leaky model ---
spend_drop_pct             0.544909
vendor_risk_score          0.436266
logins_past90d             0.004866
tenure_months              0.004481
spend_per_login            0.004286
avg_spend

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
exclusions = """
WHAT I EXCLUDED AND WHY
========================
spend_drop_pct
  reason: label-derived - built from the future spend value that defines
  churn_next_month; the AUC jump test in Section 3 confirms it.

vendor_risk_score
  reason: product/decision flag - encodes an existing system's decision,
  not underlying behavior; usable only as a baseline to beat, never as
  a model input.

raw client_id as a feature
  reason: an identifier, not a behavior - used only as the split GROUP,
  never as model input, to stop client memorization.

any future-window aggregate
  reason: would overlap or follow the label window - same failure mode
  as spend_drop_pct, generalized.

free-text fields (ticket notes, etc.)
  reason: out of scope here; also carries PII risk needing separate
  review before ever entering a model.

client names, URLs, literal queries
  reason: privacy - none appear anywhere; dataset is synthetic and
  generated in-notebook.
"""
print(exclusions)


WHAT I EXCLUDED AND WHY
spend_drop_pct
  reason: label-derived - built from the future spend value that defines
  churn_next_month; the AUC jump test in Section 3 confirms it.

vendor_risk_score
  reason: product/decision flag - encodes an existing system's decision,
  not underlying behavior; usable only as a baseline to beat, never as
  a model input.

raw client_id as a feature
  reason: an identifier, not a behavior - used only as the split GROUP,
  never as model input, to stop client memorization.

any future-window aggregate
  reason: would overlap or follow the label window - same failure mode
  as spend_drop_pct, generalized.

free-text fields (ticket notes, etc.)
  reason: out of scope here; also carries PII risk needing separate
  review before ever entering a model.

client names, URLs, literal queries
  reason: privacy - none appear anywhere; dataset is synthetic and
  generated in-notebook.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.